In [6]:
import pandas as pd

In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
df = pd.read_csv(
    "../data/processed/processed_resume.csv"
)

print(df.shape)

(2481, 6)


In [9]:
df[["ID", "Category", "processed_resume"]].head()

,ID,Category,processed_resume
0,16852973,HR,hr administrator marketing associate hr admini...
1,22323967,HR,hr specialist us hr operations summary versati...
2,33176873,HR,hr director summary 20 years experience recrui...
3,27018550,HR,hr specialist summary dedicated driven dynamic...
4,17812897,HR,hr manager skill highlights hr skills hr depar...


In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")

C:\Users\Dell\OneDrive\Desktop\AI-Resume-Screening-System\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Dell\OneDrive\Desktop\AI-Resume-Screening-System\.venv\Lib

In [11]:
job_description = """
We are looking for a Python developer with experience in
Python, SQL, MongoDB, FastAPI, REST API, React and JavaScript.
The candidate should have experience in backend development,
database management and web application development.
"""

In [12]:
job_embedding = model.encode(
    [job_description]
)

print(job_embedding.shape)

(1, 384)


In [13]:
resume_embeddings = model.encode(
    df["processed_resume"].tolist(),
    show_progress_bar=True
)

print(resume_embeddings.shape)

Batches: 100%|██████████| 78/78 [01:58<00:00,  1.52s/it]

(2481, 384)


In [14]:
semantic_scores = cosine_similarity(
    job_embedding,
    resume_embeddings
)[0]

In [15]:
semantic_percentage = semantic_scores * 100

In [16]:
semantic_percentage[:10]

array([14.63618 , 19.38861 , 19.4687  , 18.910393, 15.853694, 14.554863,
       18.193476, 17.643333, 22.835327, 16.53642 ], dtype=float32)

In [17]:
semantic_results = df[
    ["ID", "Category"]
].copy()

semantic_results["semantic_score"] = semantic_percentage

In [18]:
semantic_results = semantic_results.sort_values(
    "semantic_score",
    ascending=False
).reset_index(drop=True)

In [19]:
semantic_results["semantic_rank"] = range(
    1,
    len(semantic_results) + 1
)

In [20]:
semantic_results.head(10)

,ID,Category,semantic_score,semantic_rank
0,83816738,INFORMATION-TECHNOLOGY,44.445694,1
1,62994611,AGRICULTURE,42.507851,2
2,22351830,CONSULTANT,41.498257,3
3,99124477,BANKING,40.690754,4
4,12144825,AVIATION,39.289509,5
5,20674668,INFORMATION-TECHNOLOGY,39.182713,6
6,12011623,ENGINEERING,38.891281,7
7,85101052,DESIGNER,38.203735,8
8,43311839,CONSULTANT,36.977123,9
9,23464505,AVIATION,36.922413,10


In [21]:
semantic_results[
    [
        "semantic_rank",
        "ID",
        "Category",
        "semantic_score"
    ]
].to_csv(
    "../data/processed/semantic_matching_results.csv",
    index=False
)

In [22]:
check_results = pd.read_csv(
    "../data/processed/semantic_matching_results.csv"
)

print(check_results.shape)
check_results.head(10)

(2481, 4)


,semantic_rank,ID,Category,semantic_score
0,1,83816738,INFORMATION-TECHNOLOGY,44.445694
1,2,62994611,AGRICULTURE,42.507850
2,3,22351830,CONSULTANT,41.498257
3,4,99124477,BANKING,40.690754
4,5,12144825,AVIATION,39.289510
5,6,20674668,INFORMATION-TECHNOLOGY,39.182713
6,7,12011623,ENGINEERING,38.891280
7,8,85101052,DESIGNER,38.203735
8,9,43311839,CONSULTANT,36.977123
9,10,23464505,AVIATION,36.922413


In [23]:
print("Minimum:", semantic_results["semantic_score"].min())
print("Maximum:", semantic_results["semantic_score"].max())
print("Missing:", semantic_results["semantic_score"].isna().sum())

Minimum: -6.2579064
Maximum: 44.445694
Missing: 0


In [24]:
check_results = pd.read_csv(
    "../data/processed/semantic_matching_results.csv"
)

print(check_results.shape)
print(check_results.isna().sum())

(2481, 4)
semantic_rank     0
ID                0
Category          0
semantic_score    0
dtype: int64
